***

# **EDU Data (Edu_2, Edu_3)**

***

This file is dedicated to converting Seth's work on the Edu_2 and Edu_3 indicators from R to Python. The work Seth did is still up to date, all we want to do is make it uniform with the rest of the work we're doing.

***

## **Packages**

***

In [32]:
import pandas as pd
from tqdm import tqdm
import numpy as np

***

## **Processing**

***

In [50]:
# Datasets are from 2017-2023 and are contained in URLs

urls = [
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr23.txt",
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr22.txt",
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr21.txt",
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr20.txt",
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr19.txt",
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr18.txt",
    "https://www3.cde.ca.gov/demo-downloads/acgr/acgr17.txt"
]

# Read data from URLs into dfs
dataframes = [pd.read_csv(url, sep="\t", header=0) for url in urls]

C:\Users\jchoy\AppData\Local\Temp\ipykernel_25344\3552500611.py:14: DtypeWarning: Columns (34,35,36) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes = [pd.read_csv(url, sep="\t", header=0) for url in urls]


In [51]:
# List of col names

nms = [
    "AcYear", "AggLevel", "CountyCode", "DistCode",
    "SchoolCode", "CountyName", "DistName", "SchoolName",
    "Charter", "DASS", "RepCat", "Cohort", "HSDiploma_Ct",
    "HSDiploma_Pct", "AG_Ct", "AG_Pct", "Biliteracy_Ct",
    "Biliteracy_Pct", "GSSMerit_Ct", "GSSMerit_Pct",
    "CHSPE_Ct", "CHSPE_Pct", "AdEdDiploma_Ct",
    "AdEdDiploma_Pct", "SPED_Ct", "SPED_Pct",
    "GED_Ct", "GED_Pct", "OtherTransfer_Ct",
    "OtherTransfer_Pct", "Dropout_Ct", "Dropout_Pct",
    "StillEnrolled_Ct", "StillEnrolled_Pct"
]


# 2021-2022 does not have the same columns as others, so we drop the ones that don't match

columns_df1 = set(dataframes[2].columns)
columns_df2 = set(dataframes[1].columns)

# Identify columns in df2 not in df1

extra_columns = columns_df2 - columns_df1

# Drop extra columns from dataframes
dataframes[1] = dataframes[1].drop(columns=extra_columns)

# Renaming column names of dataframes

for df in dataframes:
    df.columns = nms

# Reversing list

dataframes.reverse()

# Initialize the combined df with the first one
#schools = dataframes[6]  # schools16_17 is the 7th DataFrame in the list
schools = dataframes[0]

# Append the remaining DataFrames
for df in dataframes[1:6]:
    schools = pd.concat([schools, df], ignore_index=True)

# Replace empty strings and '*' with NaN
schools.replace({"": np.nan, "*": np.nan}, inplace=True)

# List of SACOG counties

sacog_counties = [
    "El Dorado", "Placer", "Sacramento",
    "Sutter", "Yolo", "Yuba"
]

# List of categories

cats = ["RB", "RA", "RF", "RH", "RD", "RP", "RT", "RW", "TA", "SS"]

In [53]:
# Select categorical columns

schools_cat = schools.loc[:, "AcYear":"RepCat"]

# Convert specified columns to num

numeric_columns = ["Cohort", "HSDiploma_Ct", "HSDiploma_Pct", "AG_Ct", "AG_Pct", "Biliteracy_Ct",
                   "Biliteracy_Pct", "GSSMerit_Ct", "GSSMerit_Pct", "CHSPE_Ct", "CHSPE_Pct",
                   "AdEdDiploma_Ct", "AdEdDiploma_Pct", "SPED_Ct", "SPED_Pct", "GED_Ct", "GED_Pct",
                   "OtherTransfer_Ct", "OtherTransfer_Pct", "Dropout_Ct", "Dropout_Pct", "StillEnrolled_Ct",
                   "StillEnrolled_Pct"]

schools_num = schools.loc[:, numeric_columns].apply(pd.to_numeric, errors='coerce')

# Combine the categorical and num

schools_combined = pd.concat([schools_cat, schools_num], axis=1)

# Filter the combined df

sacog_schools = schools_combined[
    (schools_combined["CountyName"].isin(sacog_counties)) &
    (schools_combined["AggLevel"] == "D") &
    (schools_combined["Charter"] == "All") &
    (schools_combined["DASS"] == "All") &
    (schools_combined["RepCat"].isin(cats)) &
    (~schools_combined["HSDiploma_Pct"].isna())
]

# Display the filtered DataFrame

display(sacog_schools.head(5))

,AcYear,AggLevel,CountyCode,DistCode,SchoolCode,CountyName,DistName,SchoolName,Charter,DASS,...,SPED_Ct,SPED_Pct,GED_Ct,GED_Pct,OtherTransfer_Ct,OtherTransfer_Pct,Dropout_Ct,Dropout_Pct,StillEnrolled_Ct,StillEnrolled_Pct
12361,2016-17,D,9,10090.0,0.0,El Dorado,El Dorado County Office of Education,District Office,All,All,...,0.0,0.0,0.0,0.0,0.0,0.0,3.0,13.0,9.0,39.1
12363,2016-17,D,9,10090.0,0.0,El Dorado,El Dorado County Office of Education,District Office,All,All,...,0.0,0.0,1.0,1.4,2.0,2.8,9.0,12.7,16.0,22.5
12367,2016-17,D,9,10090.0,0.0,El Dorado,El Dorado County Office of Education,District Office,All,All,...,0.0,0.0,4.0,3.8,4.0,3.8,24.0,23.1,15.0,14.4
12373,2016-17,D,9,10090.0,0.0,El Dorado,El Dorado County Office of Education,District Office,All,All,...,0.0,0.0,4.0,2.4,5.0,3.0,38.0,23.2,37.0,22.6
12374,2016-17,D,9,10090.0,0.0,El Dorado,El Dorado County Office of Education,District Office,All,All,...,0.0,0.0,5.0,2.3,6.0,2.8,42.0,19.5,43.0,20.0


In [54]:
# Select and rename RepCat values for sacog_ag
sacog_ag = sacog_schools.loc[:, ["AcYear", "CountyName", "DistName", "RepCat", "AG_Ct", "AG_Pct"]]
sacog_ag['RepCat'] = sacog_ag['RepCat'].replace({
    "RB": "Black",
    "RI": "Indigenous",
    "RA": "Asian",
    "RF": "Filipino",
    "RH": "Latino",
    "RD": "Not Reported",
    "RP": "Pacific Islander",
    "RT": "Two or More",
    "RW": "White",
    "TA": "Total",
    "SS": "Socioeconomically Disadvantaged"
})

# Filter and select specific columns for sacog_county
sacog_county = schools[
    (schools["CountyName"].isin(sacog_counties)) &
    (schools["AggLevel"] == "C") &
    (schools["Charter"] == "All") &
    (schools["DASS"] == "All") &
    (schools["RepCat"].isin(cats)) &
    (~schools["HSDiploma_Pct"].isna())
]

sacog_ag_cty = sacog_county.loc[:, ["AcYear", "CountyName", "DistName", "RepCat", "AG_Ct", "AG_Pct"]]
sacog_ag_cty['RepCat'] = sacog_ag_cty['RepCat'].replace({
    "RB": "Black",
    "RI": "Indigenous",
    "RA": "Asian",
    "RF": "Filipino",
    "RH": "Latino",
    "RD": "Not Reported",
    "RP": "Pacific Islander",
    "RT": "Two or More",
    "RW": "White",
    "TA": "Total",
    "SS": "Socioeconomically Disadvantaged"
})

sacog_ag_cty['DistName'] = "County Total"

# Combine the DataFrames
sacog_combined = pd.concat([sacog_ag_cty, sacog_ag])

# Arrange by AcYear and CountyName
sacog_combined = sacog_combined.sort_values(by=["AcYear", "CountyName"])

In [57]:
sacog_combined.head(10)

,AcYear,CountyName,DistName,RepCat,AG_Ct,AG_Pct
890,2016-17,El Dorado,County Total,Asian,69,83.1
891,2016-17,El Dorado,County Total,Black,9,26.5
893,2016-17,El Dorado,County Total,Filipino,13,39.4
894,2016-17,El Dorado,County Total,Latino,96,29.2
897,2016-17,El Dorado,County Total,Two or More,47,54.0
898,2016-17,El Dorado,County Total,White,763,52.3
904,2016-17,El Dorado,County Total,Socioeconomically Disadvantaged,156,23.9
905,2016-17,El Dorado,County Total,Total,1010,49.2
12361,2016-17,El Dorado,El Dorado County Office of Education,Black,0.0,0.0
12363,2016-17,El Dorado,El Dorado County Office of Education,Latino,0.0,0.0


***

## **BEA Output_1**

***